In [1]:
"""
Code used to plot the graph showcasing results.
Each experiment appends to the existing .csv file, so we average values
across all experiments before plotting
"""

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import glob
import os


FILE_MAPPING = {
    "ae_results.csv": "Autoencoder",
    "pca_results.csv": "PCA",
    "rsvd_results.csv": "RSVD",
    "grp_results.csv": "Gaussian RP",
    "srp_results.csv": "Sparse RP",
    "jl_results.csv": "JL (Hadamard)",
}

TITLE_SIZE = 20
LABEL_SIZE = 16
TICK_SIZE = 14
LEGEND_SIZE = 14

In [2]:
def load_all_data():
    """
    Loads all .csv files, adds a 'Method' column,
    and combines them into one DataFrame.
    """
    all_dataframes = []

    csv_files = glob.glob("../results/*.csv")

    if not csv_files:
        print("No csv files found.")
        return pd.DataFrame()

    print(f"Found {len(csv_files)} result files")

    for f in csv_files:
        filename = os.path.basename(f)

        method_name = FILE_MAPPING.get(filename)

        if method_name is None:
            print(f"Warning: Skipping file '{filename}' (not in FILE_MAPPING).")
            continue

        try:
            df = pd.read_csv(f)
            df["Method"] = method_name
            all_dataframes.append(df)
            print(f"  Loaded: {filename} (as '{method_name}')")
        except Exception as e:
            print(f"Error loading {filename}: {e}")

    if not all_dataframes:
        print("Error: No valid data was loaded.")
        return pd.DataFrame()

    # Combine all individual dataframes into one big one
    full_df = pd.concat(all_dataframes, ignore_index=True)

    full_df["dimension"] = pd.to_numeric(full_df["dimension"])
    full_df["avg_time_ms"] = pd.to_numeric(full_df["avg_time_ms"])
    full_df["accuracy(%)"] = pd.to_numeric(full_df["accuracy(%)"])

    return full_df

In [3]:
def plot_accuracy_vs_dimension(df):
    """
    Generates Plot 1: Accuracy (%) vs. Dimension
    """

    plt.figure(figsize=(12, 7))
    sns.set_theme(style="whitegrid")

    plot = sns.lineplot(
        data=df,
        x="dimension",
        y="accuracy(%)",
        hue="Method",
        style="Method",
        markers=True,
        dashes=False,
        markersize=8,
        linewidth=2.5,
        estimator="mean",
        errorbar=("ci", 95),
    )

    plot.set_title("Accuracy (%) vs. Embedding Dimension (n')", fontsize=TITLE_SIZE, weight="bold")
    plot.set_xlabel("Target Embedding Dimension (Log Scale)", fontsize=LABEL_SIZE)
    plot.set_ylabel("Accuracy (%)", fontsize=LABEL_SIZE)

    plot.set_xscale("log", base=2)

    plot.invert_xaxis()

    # Set x-axis ticks to match the dimensions
    dims = sorted(df["dimension"].unique())
    plot.set_xticks(dims)
    plot.set_xticklabels(dims)

    plot.tick_params(axis='both', labelsize=TICK_SIZE)
    plot.legend(title="DR Method", fontsize=LEGEND_SIZE)
    plt.tight_layout()
    plt.savefig("accuracy_vs_dimension.png", dpi=300)
    plt.close()

In [4]:
def plot_time_vs_dimension(df):
    """
    Generates Plot 2: Time (ms) vs. Dimension
    """

    plt.figure(figsize=(12, 7))
    sns.set_theme(style="whitegrid")

    df["dimension"] = pd.to_numeric(df["dimension"], errors="coerce")
    df["accuracy(%)"] = pd.to_numeric(df["accuracy(%)"], errors="coerce")

    plot = sns.lineplot(
        data=df,
        x="dimension",
        y="avg_time_ms",
        hue="Method",
        style="Method",
        markers=True,
        dashes=False,
        markersize=8,
        linewidth=2.5,
        estimator="mean",
        errorbar=None,
        ## errorbar=("ci", 95),
    )

    plot.set_title("Matching Time (ms) vs. Embedding Dimension (n')", fontsize=TITLE_SIZE, weight="bold")
    plot.set_xlabel("Target Embedding Dimension (Log Scale)", fontsize=LABEL_SIZE)
    plot.set_ylabel("Average Matching Time (ms)", fontsize=LABEL_SIZE)

    plot.set_xscale("log", base=2)

    plot.invert_xaxis()

    # Set x-axis ticks to match the dimensions
    dims = sorted(df["dimension"].unique())
    plot.set_xticks(dims)
    plot.set_xticklabels(dims)

    plot.tick_params(axis='both', labelsize=TICK_SIZE)

    plot.legend(title="DR Method", fontsize=LEGEND_SIZE)
    plt.tight_layout()
    plt.savefig("time_vs_dimension.png")
    plt.close()

In [5]:
def plot_tradeoff_curve(df):
    """
    Generates Plot 3: EER (%) vs. Time (ms)
    """

    plt.figure(figsize=(12, 7))
    sns.set_theme(style="whitegrid")

    plot = sns.lineplot(
        data=df,
        x="avg_time_ms",
        y="EER(%)",
        hue="Method",
        style="Method",
        markers=False,
        dashes=False,
        markersize=8,
        linewidth=2.5,
    )

    plot.set_title("EER (%) vs. Matching Time (ms)", fontsize=TITLE_SIZE, weight="bold")
    plot.set_xlabel("Average Matching Time (ms)", fontsize=LABEL_SIZE)
    plot.set_ylabel("EER (%)", fontsize=LABEL_SIZE)

    plot.tick_params(axis='both', labelsize=TICK_SIZE)

    plot.legend(title="DR Method", fontsize=LEGEND_SIZE)
    plt.tight_layout()
    plt.savefig("eer_vs_time_tradeoff.png", dpi=300)
    plt.close()

In [6]:
def plot_eer_vs_dimension(df):
    plt.figure(figsize=(12, 7))
    sns.set_theme(style="whitegrid")

    plot = sns.lineplot(
        data=df,
        x="dimension",
        y="EER(%)",
        hue="Method",
        style="Method",
        markers=True,
        dashes=False,
        markersize=8,
        linewidth=2.5,
        estimator="mean",
        errorbar=("ci", 95),
    )

    plot.set_title(
        "Equal Error Rate (EER) vs. Embedding Size (n')",
        fontsize=TITLE_SIZE,
        weight="bold",
    )
    plot.set_xlabel("Target Embedding Dimension (Log Scale)", fontsize=LABEL_SIZE)
    plot.set_ylabel("Equal Error Rate (%)", fontsize=LABEL_SIZE)
    plot.set_xscale("log", base=2)

    dims = sorted(df["dimension"].unique())
    plot.set_xticks(dims)
    plot.set_xticklabels(dims)

    plot.invert_xaxis()  # Lower dimension is "more reduction"
    plot.tick_params(axis='both', labelsize=TICK_SIZE)


    plot.legend(title="EER", fontsize=LEGEND_SIZE)
    plt.tight_layout()
    plt.savefig("eer_vs_dimension.png", dpi=300)
    plt.close()


In [7]:
def main():
    df = load_all_data()

    if df.empty:
        return

    plot_accuracy_vs_dimension(df)
    plot_time_vs_dimension(df)
    plot_tradeoff_curve(df)
    plot_eer_vs_dimension(df)

if __name__ == "__main__":
    main()


Found 7 result files
  Loaded: rsvd_results.csv (as 'RSVD')
  Loaded: srp_results.csv (as 'Sparse RP')
  Loaded: ae_results.csv (as 'Autoencoder')
  Loaded: grp_results.csv (as 'Gaussian RP')
  Loaded: jl_results.csv (as 'JL (Hadamard)')
  Loaded: pca_results.csv (as 'PCA')
